In [ ]:
# =====================================================================
# Local Audio Spectrogram Transformer (AST) Classifier
# Dataset: COUGHVID-v3 (Real Data)
# Goal: classify cough windows into:
#   0 = healthy
#   1 = symptomatic
#   2 = covid_19
# =====================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# 1) Config
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Current compute device: {DEVICE}")

# >>> CHANGE THESE PATHS TO MATCH YOUR LOCAL EXTRACTED FOLDER <<<
LOCAL_DATA_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3" 

CSV_PATH = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/tabular_form/tabular_form/extracted_features_coughvid_v3.csv"
AUDIO_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/public_dataset_v3/public_dataset/"

# Audio Hyperparameters
TARGET_SR = 16000
N_MELS = 128
HOP_LENGTH = 156   # Spreads 5 seconds of audio into ~512 frames
MAX_FRAMES = 512   # Fixed sequence length for the AST temporal axis

# Training Hyperparameters
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-4

ID2LABEL = {
    0: "healthy",
    1: "symptomatic",
    2: "covid_19",
}

# -------------------------
# 2) Utility Functions
# -------------------------
def pad_or_truncate(mel, max_len):
    """Ensures all Mel spectrograms match the precise time dimension."""
    if mel.size(1) > max_len:
        return mel[:, :max_len]
    elif mel.size(1) < max_len:
        pad_amount = max_len - mel.size(1)
        return torch.nn.functional.pad(mel, (0, pad_amount))
    return mel
# -------------------------
# 3) Load Real Master Metadata & Filter by Existing Local Audio Files
# -------------------------
CSV_PATH = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/metadata_compiled.csv"

# Based on your previous errors, your actual sound files live in this nested folder:
AUDIO_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/public_dataset_v3/public_dataset/"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Could not find master metadata CSV at: {CSV_PATH}")

print(f"Parsing master COUGHVID metadata from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Clean up rows missing diagnostic status labels
label_map = {"healthy": 0, "symptomatic": 1, "COVID-19": 2}
df = df.dropna(subset=['status'])
df['label'] = df['status'].map(label_map)
df = df.dropna(subset=['label'])

# --- DEFENSIVE DISK SCANNER ---
# Check if the primary audio directory exists; fall back to the csv folder if not
if not os.path.exists(AUDIO_DIR):
    AUDIO_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/"
    print(f"[Notice] Primary directory missing. Scanning alternative folder: {AUDIO_DIR}")

print(f"Scanning local disk for available .webm files in: {AUDIO_DIR}")
local_files = os.listdir(AUDIO_DIR)

# Build an indexing set of UUIDs that are physically present on your hard drive
existing_uuids = {os.path.splitext(f)[0] for f in local_files if f.endswith('.webm')}
print(f"Found {len(existing_uuids)} real audio files on disk.")

# Cross-reference the database entries, dropping rows for files you don't have
df['uuid_str'] = df['uuid'].astype(str)
df_filtered = df[df['uuid_str'].isin(existing_uuids)].copy()

if len(df_filtered) == 0:
    raise RuntimeError(
        f"Zero matching audio files found! Please verify where your webm files are stored.\n"
        f"Sample UUID from CSV: {df['uuid'].iloc[0]}\n"
        f"Sample file found on disk: {local_files[0] if local_files else 'None'}"
    )

# Compile paths safely using only confirmed file tracks
all_files = [os.path.join(AUDIO_DIR, f"{uid}.webm") for uid in df_filtered['uuid_str']]
all_labels = df_filtered['label'].values.astype(int)

print(f"🎉 Successfully matched {len(all_files)} audio files with metadata labels!")
# -------------------------
# 4) Stratified Dataset Split
# -------------------------
X_train_files, X_temp, y_train, y_temp = train_test_split(
    all_files, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
)
X_val_files, X_test_files, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("\n--- Train/Val/Test Split Summary ---")
print(f"Train File Count : {len(X_train_files)}")
print(f"Val File Count   : {len(X_val_files)}")
print(f"Test File Count  : {len(X_test_files)}")

# -------------------------
# 5) Lazy-Loading Audio Dataset
# -------------------------
class LocalCoughvidDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.mel_transform = T.MelSpectrogram(
            sample_rate=TARGET_SR,
            n_fft=1024,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        y = self.labels[idx]
        path = self.file_paths[idx]
        
        # REMOVED TRY-EXCEPT: If it breaks, we want to know why!
        waveform, sr = torchaudio.load(path)
        
        if sr != TARGET_SR:
            resampler = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler(waveform)
            
        if waveform.size(0) > 1: 
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        mel = self.mel_transform(waveform).squeeze(0)
        mel = pad_or_truncate(mel, MAX_FRAMES)
        
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel, y

train_loader = DataLoader(LocalCoughvidDataset(X_train_files, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(LocalCoughvidDataset(X_val_files, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(LocalCoughvidDataset(X_test_files, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------------
# 6) Audio Spectrogram Transformer Architecture
# -------------------------
class AudioSpectrogramTransformer(nn.Module):
    def __init__(
        self,
        n_mels=128,
        max_frames=512,
        patch_size=(16, 16),
        num_classes=3,
        d_model=128,
        n_heads=4,
        n_layers=3,
        d_ff=256,
        dropout=0.1
    ):
        super().__init__()
        
        # Calculate standard rectangular vision grid dimensions
        f_dim = n_mels // patch_size[0]
        t_dim = max_frames // patch_size[1]
        self.num_patches = f_dim * t_dim
        
        # Non-overlapping convolution maps 2D space directly into Transformer embeddings
        self.patch_proj = nn.Conv2d(
            in_channels=1, 
            out_channels=d_model, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
        # Core learnable tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, 
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        # Input shape: [Batch, N_Mels, Time_Frames]
        x = x.unsqueeze(1) # Add Channel Dimension -> [B, 1, F, T]
        
        # Flatten map patches -> [B, d_model, num_patches]
        h = self.patch_proj(x).flatten(2).transpose(1, 2) # -> [B, num_patches, d_model]
        
        # Concat CLS Token sequence
        B = h.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        h = torch.cat((cls_tokens, h), dim=1) # -> [B, num_patches + 1, d_model]
        
        # Positional Encoding Injection
        h = h + self.pos_embed
        h = self.encoder(h)
        
        # Classify purely off the contextualized CLS marker 
        h = self.norm(h[:, 0])
        return self.cls_head(h)

model = AudioSpectrogramTransformer(
    n_mels=N_MELS,
    max_frames=MAX_FRAMES,
    patch_size=(16, 16),
    num_classes=3
).to(DEVICE)

# Calculate balancing distribution penalties for the loss function
class_counts = np.bincount(y_train, minlength=3)
class_weights = len(y_train) / (3.0 * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# -------------------------
# 7) Execution Loops
# -------------------------
def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss, y_true, y_pred = 0.0, [], []

    pbar = tqdm(loader, desc="Training" if train_mode else "Evaluating", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        if train_mode: 
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)

        y_true.extend(yb.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    return avg_loss, acc, y_true, y_pred

# -------------------------
# 8) Optimization Process
# -------------------------
best_val_acc = -1
best_state = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("\nStarting Local AST Model Training Pipeline...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, train_mode=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Save best state locally
torch.save({
    "model_state_dict": best_state,
    "history": history,
    "id2label": ID2LABEL
}, "best_ast_coughvid_local.pt")

# -------------------------
# 9) Final Evaluation Session
# -------------------------
print("\n--- Loading Best Model State for Test Set Evaluation ---")
model.load_state_dict(best_state)
model.eval()

test_loss, test_acc, y_true, y_pred = run_epoch(test_loader, train_mode=False)

# Parse macro/weighted evaluation matrices explicitly
acc = accuracy_score(y_true, y_pred)
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n[Overall Metrics]")
print(f"Accuracy           : {acc:.4f}")
print(f"Macro Precision    : {precision_macro:.4f}")
print(f"Macro Recall       : {recall_macro:.4f}")
print(f"Macro F1-score     : {f1_macro:.4f}")
print(f"Weighted Precision : {precision_weighted:.4f}")
print(f"Weighted Recall    : {recall_weighted:.4f}")
print(f"Weighted F1-score  : {f1_weighted:.4f}")

print("\n[Classification Report]")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=[ID2LABEL[0], ID2LABEL[1], ID2LABEL[2]],
    digits=4,
    zero_division=0
))

print("\n[Confusion Matrix]")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{ID2LABEL[i]}" for i in [0, 1, 2]],
    columns=[f"Pred_{ID2LABEL[i]}" for i in [0, 1, 2]]
)
print(cm_df)

# -------------------------
# 10) Plotting Performance Curves
# -------------------------
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("AST Loss Curve")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="train_acc")
    plt.plot(history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("AST Accuracy Curve")
    plt.legend()

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Matplotlib generation skipped:", e)

In [ ]:
# =====================================================================
# Local Audio Spectrogram Transformer (AST) Classifier
# Dataset: COUGHVID-v3 (Real Data)
# Goal: Classify cough windows into binary targets:
#   0 = healthy
#   1 = unhealthy (symptomatic / covid_19 combined)
# =====================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# 1) Config
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Current compute device: {DEVICE}")

# File Storage Configuration
LOCAL_DATA_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3" 
CSV_PATH = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/metadata_compiled.csv"
AUDIO_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/public_dataset_v3/public_dataset/"

# Audio Hyperparameters
TARGET_SR = 16000
N_MELS = 128
HOP_LENGTH = 156   # Spreads 5 seconds of audio into ~512 frames
MAX_FRAMES = 512   # Fixed sequence length for the AST temporal axis

# Training Hyperparameters
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-4

# Binary Classification Map
ID2LABEL = {
    0: "healthy",
    1: "unhealthy",
}

# -------------------------
# 2) Utility Functions
# -------------------------
def pad_or_truncate(mel, max_len):
    """Ensures all Mel spectrograms match the precise time dimension."""
    if mel.size(1) > max_len:
        return mel[:, :max_len]
    elif mel.size(1) < max_len:
        pad_amount = max_len - mel.size(1)
        return torch.nn.functional.pad(mel, (0, pad_amount))
    return mel

# -------------------------
# 3) Load Real Master Metadata & Filter by Existing Local Audio Files
# -------------------------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Could not find master metadata CSV at: {CSV_PATH}")

print(f"Parsing master COUGHVID metadata from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Binary Mapping: Symptomatic and COVID-19 are explicitly consolidated into 1 (unhealthy)
label_map = {"healthy": 0, "symptomatic": 1, "COVID-19": 1}
df = df.dropna(subset=['status'])
df['label'] = df['status'].map(label_map)
df = df.dropna(subset=['label'])

# --- DEFENSIVE DISK SCANNER ---
if not os.path.exists(AUDIO_DIR):
    AUDIO_DIR = "/Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/"
    print(f"[Notice] Primary directory missing. Scanning alternative folder: {AUDIO_DIR}")

print(f"Scanning local disk for available .webm files in: {AUDIO_DIR}")
local_files = os.listdir(AUDIO_DIR)

# Build an indexing set of UUIDs physically present on disk
existing_uuids = {os.path.splitext(f)[0] for f in local_files if f.endswith('.webm')}
print(f"Found {len(existing_uuids)} real audio files on disk.")

# Cross-reference database entries, dropping rows missing physical counterparts
df['uuid_str'] = df['uuid'].astype(str)
df_filtered = df[df['uuid_str'].isin(existing_uuids)].copy()

if len(df_filtered) == 0:
    raise RuntimeError(
        f"Zero matching audio files found! Please verify where your webm files are stored.\n"
        f"Sample UUID from CSV: {df['uuid'].iloc[0]}\n"
        f"Sample file found on disk: {local_files[0] if local_files else 'None'}"
    )

# Compile validated path mappings
all_files = [os.path.join(AUDIO_DIR, f"{uid}.webm") for uid in df_filtered['uuid_str']]
all_labels = df_filtered['label'].values.astype(int)

print(f"🎉 Successfully matched {len(all_files)} audio files with binary metadata labels!")

# -------------------------
# 4) Stratified Dataset Split
# -------------------------
X_train_files, X_temp, y_train, y_temp = train_test_split(
    all_files, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
)
X_val_files, X_test_files, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("\n--- Train/Val/Test Split Summary ---")
print(f"Train File Count : {len(X_train_files)}")
print(f"Val File Count   : {len(X_val_files)}")
print(f"Test File Count  : {len(X_test_files)}")

# -------------------------
# 5) Lazy-Loading Audio Dataset
# -------------------------
class LocalCoughvidDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.mel_transform = T.MelSpectrogram(
            sample_rate=TARGET_SR,
            n_fft=1024,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        y = self.labels[idx]
        path = self.file_paths[idx]
        
        waveform, sr = torchaudio.load(path)
        
        if sr != TARGET_SR:
            resampler = T.Resample(orig_freq=sr, new_freq=TARGET_SR)
            waveform = resampler(waveform)
            
        if waveform.size(0) > 1: 
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        mel = self.mel_transform(waveform).squeeze(0)
        mel = pad_or_truncate(mel, MAX_FRAMES)
        
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel, y

train_loader = DataLoader(LocalCoughvidDataset(X_train_files, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(LocalCoughvidDataset(X_val_files, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(LocalCoughvidDataset(X_test_files, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------------
# 6) Audio Spectrogram Transformer Architecture
# -------------------------
class AudioSpectrogramTransformer(nn.Module):
    def __init__(
        self,
        n_mels=128,
        max_frames=512,
        patch_size=(16, 16),
        num_classes=2, # Modified to 2 for binary projection output
        d_model=128,
        n_heads=4,
        n_layers=3,
        d_ff=256,
        dropout=0.1
    ):
        super().__init__()
        
        f_dim = n_mels // patch_size[0]
        t_dim = max_frames // patch_size[1]
        self.num_patches = f_dim * t_dim
        
        self.patch_proj = nn.Conv2d(
            in_channels=1, 
            out_channels=d_model, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, 
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1) # Add Channel Dimension -> [B, 1, F, T]
        h = self.patch_proj(x).flatten(2).transpose(1, 2) # -> [B, num_patches, d_model]
        
        B = h.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        h = torch.cat((cls_tokens, h), dim=1) 
        
        h = h + self.pos_embed
        h = self.encoder(h)
        
        h = self.norm(h[:, 0])
        return self.cls_head(h)

model = AudioSpectrogramTransformer(
    n_mels=N_MELS,
    max_frames=MAX_FRAMES,
    patch_size=(16, 16),
    num_classes=2 # Initialize for 2 target categories
).to(DEVICE)

# Calculate dynamic class weights relative to the 2 binary options
class_counts = np.bincount(y_train, minlength=2)
class_weights = len(y_train) / (2.0 * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# -------------------------
# 7) Execution Loops
# -------------------------
def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss, y_true, y_pred = 0.0, [], []

    pbar = tqdm(loader, desc="Training" if train_mode else "Evaluating", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        if train_mode: 
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)

        y_true.extend(yb.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(loader.dataset)
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    return avg_loss, acc, y_true, y_pred

# -------------------------
# 8) Optimization Process
# -------------------------
best_val_acc = -1
best_state = None
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("\nStarting Local AST Model Training Pipeline...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, train_mode=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, train_mode=False)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Save best state locally
torch.save({
    "model_state_dict": best_state,
    "history": history,
    "id2label": ID2LABEL
}, "best_ast_coughvid_local.pt")

# -------------------------
# 9) Final Evaluation Session
# -------------------------
print("\n--- Loading Best Model State for Test Set Evaluation ---")
model.load_state_dict(best_state)
model.eval()

test_loss, test_acc, y_true, y_pred = run_epoch(test_loader, train_mode=False)

acc = accuracy_score(y_true, y_pred)
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n[Overall Metrics]")
print(f"Accuracy           : {acc:.4f}")
print(f"Macro Precision    : {precision_macro:.4f}")
print(f"Macro Recall       : {recall_macro:.4f}")
print(f"Macro F1-score     : {f1_macro:.4f}")
print(f"Weighted Precision : {precision_weighted:.4f}")
print(f"Weighted Recall    : {recall_weighted:.4f}")
print(f"Weighted F1-score  : {f1_weighted:.4f}")

print("\n[Classification Report]")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=[ID2LABEL[0], ID2LABEL[1]],
    digits=4,
    zero_division=0
))

print("\n[Confusion Matrix]")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{ID2LABEL[i]}" for i in [0, 1]],
    columns=[f"Pred_{ID2LABEL[i]}" for i in [0, 1]]
)
print(cm_df)

# -------------------------
# 10) Plotting Performance Curves
# -------------------------
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("AST Loss Curve")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="train_acc")
    plt.plot(history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("AST Accuracy Curve")
    plt.legend()

    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Matplotlib generation skipped:", e)

Current compute device: cpu
Parsing master COUGHVID metadata from: /Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/metadata_compiled.csv
[Notice] Primary directory missing. Scanning alternative folder: /Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/
Scanning local disk for available .webm files in: /Users/baileys/Downloads/Intel Cup/audio_data/coughvid-v3/public_dataset_v3/coughvid_20211012/
Found 29348 real audio files on disk.
🎉 Successfully matched 17344 audio files with binary metadata labels!

--- Train/Val/Test Split Summary ---
Train File Count : 12140
Val File Count   : 2602
Test File Count  : 2602


/var/folders/08/v5yq0yq945gbsxm9zwp3dqq40000gn/T/ipykernel_5466/4219282723.py:211: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



Starting Local AST Model Training Pipeline...


Epoch 01/25 | Train Loss: 0.6952 | Train Acc: 0.5348 | Val Loss: 0.6947 | Val Acc: 0.4712


Epoch 02/25 | Train Loss: 0.6943 | Train Acc: 0.5526 | Val Loss: 0.6956 | Val Acc: 0.2571


Epoch 03/25 | Train Loss: 0.6933 | Train Acc: 0.5638 | Val Loss: 0.6949 | Val Acc: 0.7360


Epoch 04/25 | Train Loss: 0.6930 | Train Acc: 0.5456 | Val Loss: 0.7026 | Val Acc: 0.7544


Evaluating:  63%|██████▎   | 52/82 [00:25<00:14,  2.13it/s, loss=0.6924]